# Demo: `ask()` Method — Q&A and Explain Modes

This notebook demonstrates the refactored `ask()` method which operates in two modes:

1. **Q&A mode** — No data provided; the LLM answers general forecasting questions.
2. **Explain mode** — Data or a pre-computed profile is provided; the LLM explains the deterministic recommendation.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import numpy as np
import pandas as pd

from skforecast_ai import ForecastingAssistant

## Setup

Create an assistant with an LLM configured. Replace the model string with your provider.

In [3]:
# Use any supported provider: "openai:gpt-4o-mini", "anthropic:claude-sonnet-4-20250514",
# "ollama:llama3.1" (requires local Ollama), etc.
assistant = ForecastingAssistant(llm="ollama:qwen2.5:7b-instruct")

## Q&A Mode (No Data)

When no data or profile is provided, `ask()` sends the question directly to the LLM with the skforecast skills as context. The LLM answers general forecasting questions.

In [ ]:
result = assistant.ask(
    prompt="What is the difference between ForecasterRecursive and ForecasterDirect?",
    skills=[]
)

print(result.explanation)
print(f"\nProfile: {result.forecaster_profile}")
print(f"Plan: {result.plan}")

Great question! The choice between `ForecasterRecursive` and `ForecasterDirect` in skforecast depends on how you want to handle each step of your forecasting process. Here’s a concise explanation:

1. **ForecasterDirect:**
   - In direct forecasting, the model forecasts each future time point directly.
   - No rolling forecast is needed; predictions start from the end of the training data and extend into the future by adding new steps one at a time.
   - This approach is simpler and generally faster.

2. **ForecasterRecursive:**
   - Recursive forecasting involves making short-term forecasts at each step and using these as input for the next forecast.
   - It uses previously predicted values in place of actual values that might not be available when making predictions further into the future.
   - This method can lead to more accurate long-term forecasts by better capturing the dynamic behavior over time.

In summary, **ForecasterDirect** is straightforward and fast, suitable for simpl

In [ ]:
# Another Q&A example — asking about prediction intervals
result = assistant.ask(
    prompt="How do I add prediction intervals to my forecast?",
    skills=["prediction-intervals"],
)

print(result.explanation)

To add prediction intervals to your forecast using skforecast, you generally follow these steps:

1. **Fit the Forecaster**: Train your forecaster on historical data.
2. **Generate Intervals**: Use the `predict_interval` method to generate predictions along with their respective intervals.

### Step-by-Step Guide

#### 1. Fit the Forecaster

First, ensure that you have fit your forecaster (e.g., using `ForecasterRecursive`). The key step here is setting up your forecaster and ensuring it stores in-sample residuals if you're planning to use bootstrapping or conformal prediction.

```python
from skforecast.ForecasterAutoreg import ForecasterAutoreg
from sklearn.ensemble import RandomForestRegressor

# Initialize the forecaster with a Random Forest model as the regressor
forecaster = ForecasterAutoreg(
    regressor=RandomForestRegressor(n_estimators=100, random_state=123),
    lags=12  # Set the number of lagged values to use as features
)

# Fit the forecaster on your training data (y_t

## Explain Mode (With Data)

When data is provided, `ask()` first runs the deterministic pipeline (`profile()` + `generate_plan()`), then asks the LLM to explain the result in context of the user's question.

In [7]:
# Create sample data
np.random.seed(42)
dates = pd.date_range("2020-01-01", periods=365, freq="D")
df = pd.DataFrame({
    "date": dates,
    "sales": 100 + np.cumsum(np.random.randn(365)) + 10 * np.sin(np.arange(365) * 2 * np.pi / 7),
    "temperature": 20 + 10 * np.sin(np.arange(365) * 2 * np.pi / 365) + np.random.randn(365),
    "promo": np.random.choice([0, 1], size=365, p=[0.7, 0.3]),
})

df.head()

,date,sales,temperature,promo
0,2020-01-01,100.496714,19.598780,0
1,2020-01-02,108.176765,20.396226,1
2,2020-01-03,110.755418,20.356809,0
3,2020-01-04,106.868006,20.613873,0
4,2020-01-05,97.956177,19.915014,0


In [ ]:
# Ask with data — triggers profiling + plan generation + LLM explanation
result = assistant.ask(
    prompt="Explain why this forecaster and estimator were chosen.",
    data=df,
    target="sales",
    date_column="date",
    steps=14,
    skills=[]
)

print("=== LLM Explanation ===")
print(result.explanation)
print(f"\n=== Forecaster: {result.forecaster_profile.forecaster}")
print(f"=== Estimator: {result.forecaster_profile.estimator}")
print(f"=== Plan steps: {result.plan.steps}")

=== LLM Explanation ===
The choice of `ForecasterRecursive` with the `LGBMRegressor` was based on several key factors:

### 1. **Single-Series Forecasting:**
   - The task is single-series, meaning we only have one time series (sales) to predict.
   - For such tasks, recursive forecasting works well because it uses past values of the same series to make future predictions.

### 2. **LGBMRegressor as Estimator:**
   - `LGBMRegressor` is selected as the estimator due to its performance and efficiency in handling large datasets and complex features.
   - It's known for its ability to handle categorical and numerical data effectively, making it a versatile choice.
   - The chosen lags include several short-term (1, 3) and longer-term (7, 84) values. These help capture both recent trends and seasonality if present.
   - Window features (`mean` over different window sizes 7 and 91 days) allow the model to consider aggregated statistics, which can capture broader seasonal patterns.

### 3. **

## Explain Mode (With Pre-computed Profile)

If you already have a `ForecasterProfile` (from a previous `profile()` call), you can pass it directly to skip redundant profiling.

In [ ]:
# Pre-compute the profile once
profile = assistant.profile(data=df, target="sales", date_column="date")

print(f"Forecaster: {profile.forecaster}")
print(f"Estimator: {profile.estimator}")
print(f"Task type: {profile.task_type}")
print(f"Candidates: {profile.forecaster_candidates}")

In [ ]:
# Ask about the profile — no re-profiling needed
result = assistant.ask(
    prompt="Would ForecasterDirect be a better choice for this dataset? Why or why not?",
    forecaster_profile=profile,
    steps=14,
)

print(result.explanation)

## Explain Mode (With Pre-computed Plan)

You can also pass both a profile and a plan to get explanations about specific plan details.

In [ ]:
# Generate a plan with prediction intervals
plan = assistant.generate_plan(profile, steps=14, interval=[10, 90])

print(f"Interval method: {plan.interval_method}")
print(f"Lags: {plan.forecaster_kwargs.get('lags', 'N/A')}")
print(f"Use exog: {plan.use_exog}")

In [ ]:
# Ask about the specific plan — both profile and plan provided, no recomputation
result = assistant.ask(
    prompt="Explain the prediction interval configuration and what the lags mean.",
    forecaster_profile=profile,
    plan=plan,
)

print(result.explanation)

## Graceful Fallback

If the LLM is unavailable (network error, invalid API key, etc.), `ask()` still returns a result with the deterministic explanation from the plan.

In [ ]:
# Simulate a broken LLM connection
broken_assistant = ForecastingAssistant(llm="openai:nonexistent-model")

import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    result = broken_assistant.ask(
        prompt="What should I do?",
        data=df,
        target="sales",
        date_column="date",
    )

print(f"Warning issued: {w[0].message if w else 'none'}")
print(f"\nExplanation: {result.explanation}")
print(f"Plan available: {result.plan is not None}")